In [1]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import numpy as np

# Importing Train and Test Sets

In [2]:
train = pd.read_csv("C:/Users/bulut/Desktop/INDR491/imputed_train.csv")
test = pd.read_csv("C:/Users/bulut/Desktop/INDR491/imputed_test.csv")
test_label = pd.read_csv("C:/Users/bulut/Desktop/INDR491/test_data_set_labeled.csv")

In [3]:
train

,CAT1,CAT2,CAT3,CAT4,CAT5,CAT6,CAT7,CAT8,CAT9,CAT10,...,NUM19,NUM20,NUM21,NUM22,NUM23,NUM24,NUM25,NUM26,NUM27,TARGET
0,7,12,7,7,11,3,55,10,7,3,...,1.0,2.0,1.0,1.0,0.6,0.0,2.0,0.039196,0.095146,0
1,2,4,3,4,33,1,49,10,7,2,...,11.0,6.0,8.0,13.0,0.6,1.0,0.0,0.031361,0.095672,0
2,7,8,4,1,67,3,53,10,0,4,...,4.0,6.0,6.0,4.0,0.6,7.0,6.0,0.026348,0.089032,0
3,4,8,7,3,52,1,49,10,0,4,...,7.0,7.0,5.0,10.0,0.6,5.0,5.0,0.031773,0.087314,0
4,2,8,4,1,40,3,54,7,0,4,...,5.0,6.0,4.0,8.0,0.6,0.0,0.0,0.023150,0.093865,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,3,3,7,2,7,1,55,10,0,2,...,7.0,9.0,7.0,8.0,0.2,0.0,11.0,0.038615,0.096702,0
49996,2,8,7,3,78,2,55,10,7,4,...,3.0,8.0,5.0,8.0,0.2,0.0,6.0,0.036619,0.095709,0
49997,4,6,9,1,7,2,1,9,0,5,...,3.0,7.0,4.0,5.0,0.2,8.0,5.0,0.046931,0.097862,0
49998,5,1,3,4,40,1,36,10,1,2,...,11.0,11.0,12.0,12.0,0.2,5.0,12.0,0.022959,0.089737,0


In [4]:
test

,CAT1,CAT2,CAT3,CAT4,CAT5,CAT6,CAT7,CAT8,CAT9,CAT10,...,NUM18,NUM19,NUM20,NUM21,NUM22,NUM23,NUM24,NUM25,NUM26,NUM27
0,6.0,4.0,2.0,7.0,8.0,1.0,9.0,4.0,7.0,2.0,...,14.0,12.0,10.0,13.0,13.0,0.2,4.0,12.0,0.021489,0.104747
1,3.0,6.0,7.0,4.0,7.0,2.0,40.0,4.0,7.0,4.0,...,8.0,7.0,7.0,6.0,9.0,0.2,1.0,3.0,0.025130,0.094294
2,3.0,1.0,4.0,6.0,40.0,1.0,49.0,10.0,1.0,2.0,...,13.0,10.0,8.0,8.0,12.0,0.2,8.0,8.0,0.035981,0.100474
3,4.0,1.0,4.0,1.0,41.0,1.0,28.0,10.0,0.0,2.0,...,10.0,9.0,9.0,9.0,11.0,0.2,2.0,10.0,0.035098,0.098973
4,5.0,5.0,4.0,3.0,41.0,2.0,49.0,4.0,3.0,4.0,...,9.0,6.0,6.0,4.0,9.0,0.2,0.0,0.0,0.034844,0.096519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,2.0,11.0,4.0,3.0,11.0,3.0,9.0,2.0,0.0,3.0,...,8.0,1.0,2.0,1.0,6.0,0.6,3.0,0.0,0.025424,0.094813
24996,4.0,8.0,4.0,4.0,40.0,3.0,40.0,2.0,1.0,4.0,...,6.0,3.0,7.0,4.0,6.0,0.6,3.0,2.0,0.033059,0.098979
24997,3.0,12.0,4.0,4.0,33.0,1.0,49.0,10.0,0.0,3.0,...,13.0,4.0,5.0,3.0,13.0,0.6,3.0,0.0,0.033694,0.105271
24998,6.0,12.0,4.0,4.0,40.0,3.0,55.0,10.0,0.0,3.0,...,11.0,7.0,6.0,2.0,7.0,0.6,8.0,2.0,0.031074,0.094947


# Encoding the Concatenated Training and Test Set

In [5]:
categorical_cols_train = train.columns[:20] # first 20 columns are categorical
numerical_cols_train = train.columns[20:] # the rest are numerical

In [6]:
categorical_cols_test = test.columns[:20] # first 20 columns are categorical
numerical_cols_test = test.columns[20:] # the rest are numerical

In [7]:
# One-hot encode categorical columns
encoder = OneHotEncoder(handle_unknown='ignore')
train_encoded = encoder.fit_transform(train[categorical_cols_train]).toarray()
test_encoded = encoder.transform(test[categorical_cols_test]).toarray()

In [8]:
train_encoded.shape

(50000, 530)

In [9]:
test_encoded.shape

(25000, 530)

In [10]:
processed_train = pd.concat([pd.DataFrame(train_encoded, columns=encoder.get_feature_names(categorical_cols_train)), train[numerical_cols_train]], axis=1)
processed_test = pd.concat([pd.DataFrame(test_encoded, columns=encoder.get_feature_names(categorical_cols_test)), test[numerical_cols_test]], axis=1)

C:\Users\bulut\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [11]:
processed_test

,CAT1_0,CAT1_1,CAT1_2,CAT1_3,CAT1_4,CAT1_5,CAT1_6,CAT1_7,CAT2_0,CAT2_1,...,NUM18,NUM19,NUM20,NUM21,NUM22,NUM23,NUM24,NUM25,NUM26,NUM27
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,14.0,12.0,10.0,13.0,13.0,0.2,4.0,12.0,0.021489,0.104747
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,7.0,7.0,6.0,9.0,0.2,1.0,3.0,0.025130,0.094294
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,13.0,10.0,8.0,8.0,12.0,0.2,8.0,8.0,0.035981,0.100474
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,10.0,9.0,9.0,9.0,11.0,0.2,2.0,10.0,0.035098,0.098973
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,9.0,6.0,6.0,4.0,9.0,0.2,0.0,0.0,0.034844,0.096519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.0,1.0,2.0,1.0,6.0,0.6,3.0,0.0,0.025424,0.094813
24996,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,6.0,3.0,7.0,4.0,6.0,0.6,3.0,2.0,0.033059,0.098979
24997,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,13.0,4.0,5.0,3.0,13.0,0.6,3.0,0.0,0.033694,0.105271
24998,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,11.0,7.0,6.0,2.0,7.0,0.6,8.0,2.0,0.031074,0.094947


In [12]:
processed_train

,CAT1_0,CAT1_1,CAT1_2,CAT1_3,CAT1_4,CAT1_5,CAT1_6,CAT1_7,CAT2_0,CAT2_1,...,NUM19,NUM20,NUM21,NUM22,NUM23,NUM24,NUM25,NUM26,NUM27,TARGET
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,2.0,1.0,1.0,0.6,0.0,2.0,0.039196,0.095146,0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,11.0,6.0,8.0,13.0,0.6,1.0,0.0,0.031361,0.095672,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,4.0,6.0,6.0,4.0,0.6,7.0,6.0,0.026348,0.089032,0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,7.0,7.0,5.0,10.0,0.6,5.0,5.0,0.031773,0.087314,0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,6.0,4.0,8.0,0.6,0.0,0.0,0.023150,0.093865,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,9.0,7.0,8.0,0.2,0.0,11.0,0.038615,0.096702,0
49996,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,8.0,5.0,8.0,0.2,0.0,6.0,0.036619,0.095709,0
49997,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,3.0,7.0,4.0,5.0,0.2,8.0,5.0,0.046931,0.097862,0
49998,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,11.0,11.0,12.0,12.0,0.2,5.0,12.0,0.022959,0.089737,0


# Train the Model

In [13]:
# separate target variable and features
X_train = processed_train.drop('TARGET', axis=1)
y_train = processed_train['TARGET']

X_test = processed_test
y_test = test_label['TARGET']

In [14]:
y_test

1        0
2        0
3        0
4        0
5        1
        ..
24996    0
24997    0
24998    0
24999    0
25000    0
Name: TARGET, Length: 25000, dtype: int64

In [15]:
X_test.shape

(25000, 557)

In [16]:
X_train.shape

(50000, 557)

In [17]:
y_train.shape

(50000,)

# Logistic Regression

In [19]:
from sklearn.linear_model import LogisticRegression

In [29]:
# define parameter grid for logistic regression
param_grid = {
    'penalty': ['l1'],
    'C':[0.1],
    'solver': ['saga']
}

"""
# define parameter grid for logistic regression
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C':[0.001, 0.01, 0.1, 1, 10, 100],
    #'C': [i/10 for i in range(1, 100+1)]
    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
}
"""

"\n# define parameter grid for logistic regression\nparam_grid = {\n    'penalty': ['l1', 'l2', 'elasticnet'],\n    'C':[0.001, 0.01, 0.1, 1, 10, 100],\n    #'C': [i/10 for i in range(1, 100+1)]\n    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']\n}\n"

In [30]:
# define stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5)

In [31]:
# define logistic regression model
logreg = LogisticRegression()

In [32]:
# perform grid search to find best hyperparameters
grid_search = GridSearchCV(logreg, param_grid=param_grid, cv=skf, scoring='roc_auc')
grid_search.fit(X_train, y_train)

C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:352: ConvergenceWarning: The max_iter 

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(),
             param_grid={'C': [0.1], 'penalty': ['l1'], 'solver': ['saga']},
             scoring='roc_auc')

In [33]:
# print best hyperparameters
print("Best hyperparameters:", grid_search.best_params_)

Best hyperparameters: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}


In [34]:
# make predictions on test data
y_pred = grid_search.predict_proba(X_test)[:,1]

In [38]:
y_pred

array([0.05322172, 0.09740225, 0.06702609, ..., 0.03979068, 0.06919471,
       0.12483878])

In [36]:
test_auc = roc_auc_score(y_test, y_pred)
print("AUC-ROC score for test data:", test_auc)

AUC-ROC score for test data: 0.7584566883945606


In [36]:
"""
# export it as .txt file
output = pd.DataFrame({'TARGET': y_pred})
output.to_csv('C:/Users/bulut/Desktop/logistic_regression_predicted_y.txt', sep='\t', index=False)
"""

# SVM / SVC

In [18]:
from sklearn.svm import LinearSVC

In [19]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'loss': ['hinge', 'squared_hinge'],
}

In [20]:
# define stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5)

In [21]:
# Linear SVC
# define Linear SVC model
svc = LinearSVC()

In [22]:
# perform grid search to find best hyperparameters
grid_search = GridSearchCV(svc, param_grid=param_grid, cv=skf, scoring='roc_auc')
grid_search.fit(X_train, y_train)

C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\bulut\anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number 

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LinearSVC(),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'loss': ['hinge', 'squared_hinge'],
                         'penalty': ['l1', 'l2']},
             scoring='roc_auc')

In [23]:
# print best hyperparameters
print("Best hyperparameters:", grid_search.best_params_)

Best hyperparameters: {'C': 0.001, 'loss': 'squared_hinge', 'penalty': 'l2'}


In [25]:
# use best hyperparameters to fit the model and predict on test data
svc_best = LinearSVC(**grid_search.best_params_)
svc_best.fit(X_train, y_train)
#y_pred = svc_best.predict_proba(X_test)[:, 1]
y_pred = svc_best.decision_function(X_test)

In [26]:
test_auc = roc_auc_score(y_test, y_pred)
print("AUC-ROC score for test data:", test_auc)

AUC-ROC score for test data: 0.7581897321044903


# Random Forest

Can sklearn random forest directly handle categorical features?

No, there isn't. 
Somebody's working on this and the patch might be merged into 
mainline some day, but right now there's no support for 
categorical variables in scikit-learn except dummy (one-hot) encoding.

In [27]:
from sklearn.ensemble import RandomForestClassifier

In [32]:
# define parameter grid for random forest
param_grid = {
    #'n_estimators': [100, 200, 300],
    #'max_depth': [5, 10, 20, None],
    #'min_samples_split': [2, 5, 10],
    #'min_samples_leaf': [1, 2, 4]
    'n_estimators': [100,300],
    'max_depth': [5,20,None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [33]:
# define stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5)

In [34]:
# define random forest model
rf = RandomForestClassifier()

In [35]:
# perform grid search to find best hyperparameters
grid_search = GridSearchCV(rf, param_grid=param_grid, cv=skf, scoring='roc_auc')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=RandomForestClassifier(),
             param_grid={'max_depth': [5], 'n_estimators': [100]},
             scoring='roc_auc')

In [36]:
# use best hyperparameters to fit the model and predict on test data
rf_best = RandomForestClassifier(**grid_search.best_params_)
rf_best.fit(X_train, y_train)
y_pred = rf_best.predict_proba(X_test)[:, 1]

In [37]:
test_auc = roc_auc_score(y_test, y_pred)
print("AUC-ROC score for test data:", test_auc)

AUC-ROC score for test data: 0.7297878733397912


# Gradient Boosting Classifier

In [38]:
from sklearn.ensemble import GradientBoostingClassifier

In [39]:
# define parameter grid for extra boost classifier
param_grid = {
    #'n_estimators': [50, 100, 150, 200],
    #'learning_rate': [0.001, 0.01, 0.1, 1],
    #'max_depth': [3, 5, 7],
    #'min_samples_split': [2, 5, 10],
    #'min_samples_leaf': [1, 2, 4],
    #'max_features': ['auto', 'sqrt', 'log2']
    'n_estimators': [50,150],
    'learning_rate': [0.001,0.1,1],
    'max_depth': [3,5,7],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,4],
    'max_features': ['auto', 'sqrt', 'log2']
}

In [40]:
# define stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5)

In [41]:
# define extra boost classifier model
clf = GradientBoostingClassifier()

In [42]:
# perform grid search to find best hyperparameters
grid_search = GridSearchCV(clf, param_grid=param_grid, cv=skf, scoring='roc_auc')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=GradientBoostingClassifier(),
             param_grid={'learning_rate': [0.001], 'max_depth': [3],
                         'max_features': ['auto', 'sqrt', 'log2'],
                         'min_samples_leaf': [1], 'min_samples_split': [2],
                         'n_estimators': [50]},
             scoring='roc_auc')

In [43]:
# print best hyperparameters
print("Best hyperparameters:", grid_search.best_params_)

Best hyperparameters: {'learning_rate': 0.001, 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}


In [44]:
# fit the model with best hyperparameters and predict on test data
best_gb_clf = GradientBoostingClassifier(**grid_search.best_params_)
best_gb_clf.fit(X_train, y_train)
y_pred = best_gb_clf.predict_proba(X_test)[:,1]

In [45]:
test_auc = roc_auc_score(y_test, y_pred)
print("AUC-ROC score for test data:", test_auc)

AUC-ROC score for test data: 0.7271410235928281
